# Sensor SSL — Google Colab
อัปโหลดไฟล์ `sensor-ssl-source.zip` ที่ได้รับ แล้วรันตามลำดับ Notebook นี้เป็นทางเลือกสำหรับรันซ้ำบน Colab; ผลที่ส่งมอบรันบน CPU ในเครื่อง ไม่ใช่ผล Colab

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile
uploaded=files.upload()
name=next(k for k in uploaded if k.endswith(".zip"))
with zipfile.ZipFile(name) as z:
    for item in z.infolist():
        target=Path(item.filename)
        assert not target.is_absolute() and ".." not in target.parts
    z.extractall("/content")
%cd /content/sensor-ssl

In [ ]:
%pip install -r requirements.txt
!python -m unittest discover -v

## ดาวน์โหลดข้อมูล mirror เดียวกับการทดลอง
ตรวจ hash ก่อนใช้งาน ข้อมูลไม่ถูกรวมใน ZIP เพื่อไม่เพิ่มขนาดไฟล์

In [ ]:
import hashlib,json,urllib.request
from pathlib import Path
p=Path("data");p.mkdir(exist_ok=True)
meta=json.loads(Path("docs/DATA_PROVENANCE.json").read_text())
urllib.request.urlretrieve(meta["download_url"],p/"uci_har.zip")
assert hashlib.sha256((p/"uci_har.zip").read_bytes()).hexdigest()==meta["sha256"]
(p/"provenance.json").write_text(json.dumps(meta,indent=2))

## ทดลองหนึ่งคู่ก่อน
หากรันซ้ำ ให้เปลี่ยนชื่อ output เพื่อไม่เขียนทับผลเดิม

In [ ]:
!python -m src.experiment --mode supervised --fraction 0.1 --output runs/colab_baseline
!python -m src.experiment --mode ssl --mask both --fraction 0.1 --output runs/colab_ssl

## รันชุดทดลองทั้งหมด
ใช้เวลามากกว่าเซลล์ก่อนหน้า หากเลือก GPU ให้เพิ่ม `--device cuda` และระบุอุปกรณ์ในรายงาน

In [ ]:
!python run_matrix.py --runs runs/colab_full --epochs 30 --pretrain-epochs 30

In [ ]:
!python -m src.summarize runs/colab_full --output results/colab_summary.csv
from google.colab import files
files.download("results/colab_summary.csv")